# OU estimation of the regression residual

The residual is modelled through the exact AR(1) form of an Ornstein-Uhlenbeck process, `X_{t+1} = a + b X_t + eta_t`, fitted by OLS over a trailing window of M residuals. Only fits with `0 < b < 1` receive `theta`, `mu`, `sigma` and a half-life; every other fit keeps its state (`non_positive`, `unit_root`, `explosive`, `near_unit_root`, `degenerate`) so nothing is forced into an OU reading.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.models` and `xauusd_quant.research` and read back from `results/ou_research/<tf>/regression_<N>/ou_<M>/`. Generate those first:

```
xq ou-research --timeframe 5m --regression-window 128 --ou-window 256
xq ou-research --timeframe 5m --all-windows
xq ou-research --all
```

Three series are always shown together: the XAUUSD residual, a **detrended random walk** (the same pipeline on data with no reversion at all), and an **exact OU** with the fitted parameters. Detrending alone makes any series look OU-like, so a residual number means something only relative to the control; the reference shows what estimation noise and small-window bias produce on their own.

No trading rule is defined or evaluated anywhere. No costs are applied. Nothing here is a signal.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.models import load_ou_config

ou = load_ou_config(ROOT / "config" / "ou.yaml")

TIMEFRAME = "5m"          # change and re-run
REGRESSION_WINDOW = 128   # N: the residual being modelled
OU_WINDOW = 256           # M: residual observations per rolling fit

results = ou.results_dir(TIMEFRAME, REGRESSION_WINDOW, OU_WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} N={REGRESSION_WINDOW} M={OU_WINDOW}. Run:  "
        f"xq ou-research --timeframe {TIMEFRAME} --regression-window {REGRESSION_WINDOW} "
        f"--ou-window {OU_WINDOW}"
    )

def table(name):
    """Load a previously generated OU table (None if it was not written)."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = results / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))

summary = json.loads((results / "summary.json").read_text(encoding="utf-8"))
model = summary["model_summary"]
lineage = summary["provenance"]["dataset"]
print(f"span       : {model['first_timestamp']} -> {model['last_timestamp']}")
print(f"residuals  : {model['observations']:,}")
print(f"ticks      : {lineage['tick_dataset_version']} ({lineage['tick_dataset_status']})")
print(f"bars       : {lineage.get('bar_dataset_version')}")
if lineage.get("label"):
    print(f"WARNING    : {lineage['label']}")
for warning in summary.get("warnings", []):
    print(f"WARNING    : {warning}")

## The headline, beside its control and reference

In [ ]:
rows = []
for source in ("residual", "control_random_walk", "reference_ou"):
    block = model if source == "residual" else (model.get(source) or {})
    rows.append({"series": source, **{k: block.get(k) for k in (
        "valid_ou_fraction", "median_b", "median_theta", "median_half_life_bars",
        "half_life_p25", "half_life_p75", "median_ou_r_squared", "innovation_lag1_acf")}})
pl.DataFrame(rows)

## How each window was classified

In [ ]:
table('b_classification')

## Whole-sample static fits (a reference point only)

Pooling the entire history into one estimate assumes one stable market structure. It is shown for scale; the stability notebook is where the answer is.

In [ ]:
table('static_fit')

## Distribution of every rolling estimate

In [ ]:
table('parameter_distribution')

## Equilibrium: estimated, never forced to zero

In [ ]:
table('equilibrium')

## The OU Z-score beside the Prompt #3 rolling residual Z-score

Different questions, kept in different columns - compared, never substituted.

In [ ]:
table('zscore_comparison')

In [ ]:
plot('rolling_b')
plot('ou_equilibrium')
plot('ou_zscore')